# 12 · Data Definition Language (DDL)

DDL creates and changes the **structure** of the database.
- `CREATE TABLE`, data types, `PRIMARY KEY`
- constraints: `NOT NULL`, `UNIQUE`, `CHECK`, `DEFAULT`, `FOREIGN KEY`
- `ALTER TABLE`
- `DROP TABLE`

> These cells create their own throwaway tables (named `demo_*`) so they never
> touch the course data. Each is dropped first, so you can re-run safely.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## SQLite data types
SQLite is flexible; the core storage classes are `INTEGER`, `REAL`, `TEXT`,
`BLOB`, and `NULL`. You'll also see `NUMERIC`. Dates are stored as `TEXT`
(ISO strings) or numbers.

## `CREATE TABLE` with constraints

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_students;
CREATE TABLE demo_students (
    student_id INTEGER PRIMARY KEY,          -- auto-increments in SQLite
    full_name  TEXT NOT NULL,                -- required
    email      TEXT UNIQUE,                  -- no duplicates
    age        INTEGER CHECK (age >= 0),     -- validated
    status     TEXT NOT NULL DEFAULT 'active'-- default value
);

Insert a couple of rows and read them back:

In [ ]:
%%sql
INSERT INTO demo_students (full_name, email, age) VALUES
    ('Ada Lovelace', 'ada@school.edu', 28),
    ('Alan Turing',  'alan@school.edu', 30);
SELECT * FROM demo_students;

Notice `status` defaulted to 'active' and `student_id` auto-filled. A `CHECK` violation is rejected — this next cell **should error** (that's the point):

In [ ]:
%%sql
INSERT INTO demo_students (full_name, age) VALUES ('Bad Age', -5);

## `ALTER TABLE` — add a column

In [ ]:
%%sql
ALTER TABLE demo_students ADD COLUMN gpa REAL;
SELECT student_id, full_name, gpa FROM demo_students;

## `FOREIGN KEY`
Link an enrollments table to students:

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_enrollments;
CREATE TABLE demo_enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id    INTEGER NOT NULL,
    course        TEXT NOT NULL,
    FOREIGN KEY (student_id) REFERENCES demo_students(student_id)
);
INSERT INTO demo_enrollments (student_id, course) VALUES (1, 'Databases'), (2, 'Logic');
SELECT * FROM demo_enrollments;

## `DROP TABLE` — clean up our demo tables

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_enrollments;
DROP TABLE IF EXISTS demo_students;
SELECT 'cleaned up' AS status;

## Practice

**✏️ Exercise 1.** Create a table `demo_books` with book_id (primary key), title (required text), and price (real, must be >= 0). Then insert one row and select it.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_books;
CREATE TABLE demo_books (
    book_id INTEGER PRIMARY KEY,
    title   TEXT NOT NULL,
    price   REAL CHECK (price >= 0)
);
INSERT INTO demo_books (title, price) VALUES ('SQL for All', 24.99);
SELECT * FROM demo_books;

### ✅ Recap
DDL shapes the schema: `CREATE TABLE` with typed columns and constraints
(`NOT NULL`, `UNIQUE`, `CHECK`, `DEFAULT`, `FOREIGN KEY`), `ALTER TABLE` to
change it, `DROP TABLE` to remove it.

**Next:** `13_dml_insert_update_delete.ipynb`.